# Build Tree Diagnostics

Walk-forward TreeSHAP for tuned XGB and LGBM. At every refit `i` we:

1. Fit a fresh model on the rolling training window (mirrors `src.scaling.run_backtest`).
2. Compute `TreeExplainer(model_i).shap_values(X_oos_row_i)` and accumulate.
3. Every `INTERACTION_STRIDE` refits, also compute `shap_interaction_values` on that one OOS row.
4. Capture `booster.trees_to_dataframe()` for tree-topology plots.

Output per entry, into `results/diagnostics/<entry_id>/`:

- `tree_structures.parquet`, `tree_depth_histogram.png`, `feature_split_count.png`
- `shap_long.parquet`, `shap_interactions_long.parquet`
- `shap_summary_bar.png`, `shap_group_bar.png`, `shap_yearly_heatmap.png`, `shap_rank_stability.png`
- `shap_interaction_heatmap.png`, `shap_dependence_<feat1..3>.png`
- `tree_story_stats.json` (methodology + headline numbers)

Methodology references: Lundberg, Erion, Lee (2018) TreeSHAP; Lundberg et al. (Nature MI 2020) interaction values; Gu, Kelly, Xiu (2020 RFS) walk-forward pooled SHAP; Aas, Jullum, Løland (2021) on correlated-feature caveats. The original 18:30/Sunday-open hook is *not* a narrative anchor here — it was an upstream aggregation artifact.

In [ ]:
import os
from pathlib import Path

MANIFEST = os.environ.get("TREE_DIAG_MANIFEST", "results/MANIFEST.json")
OUT_ROOT = os.environ.get("TREE_DIAG_OUT_ROOT", "results/diagnostics")
ENTRY = os.environ.get("TREE_DIAG_ENTRY", "")  # "" = both xgb_har_tuned and lgbm_har_tuned
INTERACTION_STRIDE = int(os.environ.get("TREE_DIAG_INTERACTION_STRIDE", "65"))  # ~weekly at 13 periods/day
SKIP_EXISTING = os.environ.get("SKIP_EXISTING", "0") == "1"
# Smoke-test cap: limit OOS rows after train_window. 0 = no cap.
TREE_DIAG_END = int(os.environ.get("TREE_DIAG_END", "0"))
TRAIN_WINDOW_DAYS = int(os.environ.get("TREE_DIAG_TRAIN_WINDOW", "500"))
DATA_PATH = os.environ.get("TREE_DIAG_DATA_PATH", "all30min")

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

REPO = Path.cwd()
while REPO.parent != REPO and not (REPO / "src").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import os as _os

_os.chdir(REPO)

from src.evaluation import (  # noqa: E402
    plot_feature_split_count,
    plot_shap_dependence,
    plot_shap_group_bar,
    plot_shap_interaction_heatmap,
    plot_shap_rank_stability,
    plot_shap_summary_bar,
    plot_shap_yearly_heatmap,
    plot_tree_depth_histogram,
)
from src.executor import _build_har_and_calendar, load_and_transform  # noqa: E402
from src.transforms import PERIODS_PER_DAY, apply_horizon_shift, resolve_har_lags  # noqa: E402

# MANIFEST keys are short (xgb/lgbm); the executor module names are long
# (xgboost/lightgbm). This map is the temporary bridge documented in plan
# section "Implementation-time notes".
METHOD_MAP = {"xgb": "xgboost", "lgbm": "lightgbm"}

# Tree data-prep invariants — inline literals matching src/ml_xgboost.py and
# src/ml_lightgbm.py run() (B2 dropped the src.executor ExecutorConfig/CONFIGS
# registry). XGB and LGBM share the same data-prep and a refit cadence of 1.
_TREE_PREP = dict(
    add_calendar=True,
    target_use_diurnal=True,
    target_winsor_window=240,
    dropna_with_exog=False,
    refit_frequency=1,
)
TREE_DATA_PREP = {"xgboost": dict(_TREE_PREP), "lightgbm": dict(_TREE_PREP)}


def entry_id(entry: dict) -> str:
    return f"{entry['method']}_{entry['feature_set']}_{entry['config']}"


TREE_ENTRIES = ["xgb_har_tuned", "lgbm_har_tuned"]

In [ ]:
def load_xy(method_long: str, data_path: str) -> tuple[np.ndarray, np.ndarray, pd.DatetimeIndex, list[str]]:
    """Reproduce X, y, dates, feature_names exactly as the executor sees them.

    Mirrors :func:`src.executor.run_executor` data prep without the segment / Ridge branches.
    """
    cfg = TREE_DATA_PREP[method_long]
    df, _adj_exog = load_and_transform(
        data_path,
        exog_cols=[],
        target_use_diurnal=cfg["target_use_diurnal"],
        target_winsor_window=cfg["target_winsor_window"],
        dropna_with_exog=cfg["dropna_with_exog"],
    )
    df, feature_names = _build_har_and_calendar(df, exog_cols=[], add_calendar=cfg["add_calendar"])
    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)
    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)
    X, y, dates, _baselines = apply_horizon_shift(X, y, dates, baselines, horizon=1)
    return X, y, pd.DatetimeIndex(dates), feature_names

In [ ]:
def _make_model(method_long: str, params: dict):
    if method_long == "xgboost":
        return XGBRegressor(**{k: v for k, v in params.items() if not k.startswith("_")})
    if method_long == "lightgbm":
        return LGBMRegressor(**{k: v for k, v in params.items() if not k.startswith("_")})
    raise ValueError(f"unsupported method: {method_long}")


def _xgb_depths(df: pd.DataFrame) -> np.ndarray:
    """Walk Yes/No edges per Tree to assign each node a depth."""
    out = np.zeros(len(df), dtype=np.int32)
    for tree_id, sub in df.groupby("Tree"):
        idx_by_id = {row["ID"]: i for i, row in sub.iterrows()}
        # BFS from root (ID = f"{tree}-0")
        root_id = f"{tree_id}-0"
        if root_id not in idx_by_id:
            continue
        stack = [(root_id, 0)]
        while stack:
            nid, depth = stack.pop()
            row = df.loc[idx_by_id[nid]]
            out[idx_by_id[nid]] = depth
            for ch_col in ("Yes", "No"):
                ch = row.get(ch_col)
                if isinstance(ch, str) and ch in idx_by_id:
                    stack.append((ch, depth + 1))
    return out


def _trees_dataframe(model, refit_id: int) -> pd.DataFrame:
    if isinstance(model, XGBRegressor):
        df = model.get_booster().trees_to_dataframe().copy()
        df["node_depth"] = _xgb_depths(df)
    else:
        df = model.booster_.trees_to_dataframe().copy()
    df["refit_id"] = refit_id
    return df


def walk_forward_shap(
    method_long: str,
    params: dict,
    X: np.ndarray,
    y: np.ndarray,
    dates: pd.DatetimeIndex,
    feature_names: list[str],
    train_win: int,
    refit_frequency: int = 1,
    interaction_stride: int = 65,
    end: int | None = None,
    capture_trees_every: int = 250,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, np.ndarray]:
    """Walk-forward refit with per-step SHAP capture.

    Mirrors the predicate in :func:`src.scaling.run_backtest` (`(t - train_win + 1) % refit_frequency == 0`)
    so produced yhats are bit-identical when `compute_shap` is irrelevant.

    Returns
    -------
    shap_long : pd.DataFrame
        Columns ``[date, horizon, *feature_names]``; one row per OOS step.
    interactions_long : pd.DataFrame
        Columns ``[date, feat_i, feat_j, interaction_value]``; one row per (i,j) pair per stride hit.
    tree_structures : pd.DataFrame
        Concatenation of `trees_to_dataframe()` outputs, keyed by ``refit_id``.
    yhats : np.ndarray
        OOS predictions of length ``min(end, n) - train_win``.
    """
    n_samples = len(X)
    n_end = n_samples if not end else min(end, n_samples)
    n_oos = n_end - train_win
    yhats = np.full(n_oos, np.nan)
    shap_rows = []
    inter_rows = []
    tree_frames: list[pd.DataFrame] = []

    model = _make_model(method_long, params)
    model.fit(X[:train_win], y[:train_win])
    explainer = shap.TreeExplainer(model)
    if capture_trees_every > 0:
        tree_frames.append(_trees_dataframe(model, refit_id=0))

    refit_id = 0
    for k, t in enumerate(range(train_win, n_end)):
        x_t = X[t : t + 1]
        yhats[k] = float(model.predict(x_t)[0])

        sv = explainer.shap_values(x_t)
        if isinstance(sv, list):
            sv = sv[0]
        shap_rows.append((dates[t], 1, sv.ravel().tolist()))

        if k % interaction_stride == 0:
            iv = explainer.shap_interaction_values(x_t)
            if isinstance(iv, list):
                iv = iv[0]
            iv = np.asarray(iv).reshape(len(feature_names), len(feature_names))
            for i, fi in enumerate(feature_names):
                for j, fj in enumerate(feature_names):
                    if j < i:
                        continue
                    inter_rows.append((dates[t], fi, fj, float(iv[i, j])))

        if (t - train_win + 1) % refit_frequency == 0 and (t + 1) < n_end:
            refit_id += 1
            model = _make_model(method_long, params)
            model.fit(X[t - train_win + 1 : t + 1], y[t - train_win + 1 : t + 1])
            explainer = shap.TreeExplainer(model)
            if capture_trees_every > 0 and refit_id % capture_trees_every == 0:
                tree_frames.append(_trees_dataframe(model, refit_id))

    shap_arr = np.array([r[2] for r in shap_rows], dtype=np.float64)
    shap_long = pd.DataFrame(shap_arr, columns=feature_names)
    shap_long.insert(0, "horizon", [r[1] for r in shap_rows])
    shap_long.insert(0, "date", [r[0] for r in shap_rows])

    interactions_long = pd.DataFrame(inter_rows, columns=["date", "feat_i", "feat_j", "interaction_value"])
    tree_structures = pd.concat(tree_frames, ignore_index=True) if tree_frames else pd.DataFrame()
    return shap_long, interactions_long, tree_structures, yhats

In [ ]:
def _classify_feature(name: str) -> str:
    """Group features for the correlated-feature-caveat group bar plot."""
    n = name.lower()
    if n.startswith("adj_rv") or "_lag" in n or n.startswith("rv") or n.startswith("har_"):
        return "RV-lag"
    if any(k in n for k in ("hour", "dow", "day_of_week", "slot", "calendar")):
        return "calendar"
    if n.startswith("adj_"):
        return "exog"
    return "other"


def aggregate_shap(
    shap_long: pd.DataFrame,
    interactions_long: pd.DataFrame,
    feature_names: list[str],
) -> dict:
    """Reduce per-row SHAP into the panel of summaries the writeup needs."""
    abs_shap = shap_long[feature_names].abs()
    global_importance = (
        abs_shap.mean()
        .rename("mean_abs_shap")
        .rename_axis("feature")
        .reset_index()
        .sort_values("mean_abs_shap", ascending=False)
    )

    years = pd.to_datetime(shap_long["date"]).dt.year
    yearly_importance = (
        abs_shap.assign(year=years.values)
        .groupby("year")
        .mean()
        .stack()
        .rename("mean_abs_shap")
        .rename_axis(["year", "feature"])
        .reset_index()
    )

    n_oos = len(shap_long)
    fold_size = max(n_oos // 20, 100)
    fold_id = np.arange(n_oos) // fold_size
    perfold_importance = (
        abs_shap.assign(refit_id=fold_id)
        .groupby("refit_id")
        .mean()
        .stack()
        .rename("mean_abs_shap")
        .rename_axis(["refit_id", "feature"])
        .reset_index()
    )

    groups = {f: _classify_feature(f) for f in feature_names}
    group_importance = (
        global_importance.assign(group=global_importance["feature"].map(groups))
        .groupby("group")["mean_abs_shap"]
        .sum()
        .reset_index()
    )

    if len(interactions_long):
        top_interactions = (
            interactions_long.assign(mean_abs_interaction=interactions_long["interaction_value"].abs())
            .groupby(["feat_i", "feat_j"])["mean_abs_interaction"]
            .mean()
            .reset_index()
        )
        # Mirror to the lower triangle so heatmap is symmetric.
        mirror = top_interactions.rename(columns={"feat_i": "feat_j", "feat_j": "feat_i"})
        top_interactions = pd.concat([top_interactions, mirror], ignore_index=True).drop_duplicates(
            subset=["feat_i", "feat_j"]
        )
    else:
        top_interactions = pd.DataFrame(columns=["feat_i", "feat_j", "mean_abs_interaction"])

    pivot = perfold_importance.pivot(index="refit_id", columns="feature", values="mean_abs_shap")
    if len(pivot) >= 2:
        ranks = pivot.rank(axis=1, ascending=False)
        rhos = []
        for i in range(len(ranks) - 1):
            r1, r2 = ranks.iloc[i].values, ranks.iloc[i + 1].values
            rho = np.corrcoef(np.argsort(r1), np.argsort(r2))[0, 1]
            if np.isfinite(rho):
                rhos.append(float(rho))
        rank_stability_rho = float(np.mean(rhos)) if rhos else float("nan")
    else:
        rank_stability_rho = float("nan")

    return {
        "global_importance": global_importance,
        "yearly_importance": yearly_importance,
        "perfold_importance": perfold_importance,
        "group_importance": group_importance,
        "top_interactions": top_interactions,
        "rank_stability_rho": rank_stability_rho,
    }

In [ ]:
def _save(fig, path: Path) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def render_tree_diagnostics(
    out_dir: Path,
    label: str,
    tree_structures: pd.DataFrame,
    shap_long: pd.DataFrame,
    interactions_long: pd.DataFrame,
    aggs: dict,
    X_df: pd.DataFrame,
    feature_names: list[str],
    methodology: dict,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(tree_structures):
        tree_structures.to_parquet(out_dir / "tree_structures.parquet", index=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        plot_tree_depth_histogram(tree_structures, ax, title=f"{label}: tree-node depth")
        _save(fig, out_dir / "tree_depth_histogram.png")

        fig, ax = plt.subplots(figsize=(8, 6))
        plot_feature_split_count(tree_structures, ax, top_n=15, title=f"{label}: top-15 split counts")
        _save(fig, out_dir / "feature_split_count.png")

    shap_long.to_parquet(out_dir / "shap_long.parquet", index=False)
    if len(interactions_long):
        interactions_long.to_parquet(out_dir / "shap_interactions_long.parquet", index=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    plot_shap_summary_bar(aggs["global_importance"], ax, top_n=15, title=f"{label}: top-15 mean |SHAP|")
    _save(fig, out_dir / "shap_summary_bar.png")

    fig, ax = plt.subplots(figsize=(6, 4))
    plot_shap_group_bar(aggs["group_importance"], ax, title=f"{label}: SHAP by feature group")
    _save(fig, out_dir / "shap_group_bar.png")

    if len(aggs["yearly_importance"]):
        fig, ax = plt.subplots(figsize=(10, 6))
        plot_shap_yearly_heatmap(aggs["yearly_importance"], ax, top_n=15, title=f"{label}: yearly mean |SHAP|")
        _save(fig, out_dir / "shap_yearly_heatmap.png")

    if len(aggs["perfold_importance"]):
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_shap_rank_stability(
            aggs["perfold_importance"],
            ax,
            top_n=10,
            title=f"{label}: top-10 feature ranks across folds",
        )
        _save(fig, out_dir / "shap_rank_stability.png")

    if len(aggs["top_interactions"]):
        fig, ax = plt.subplots(figsize=(8, 7))
        plot_shap_interaction_heatmap(
            aggs["top_interactions"],
            ax,
            top_n=10,
            title=f"{label}: top-10 interaction strengths",
        )
        _save(fig, out_dir / "shap_interaction_heatmap.png")

        top_feats = aggs["global_importance"]["feature"].head(3).tolist()
        for feat in top_feats:
            partner = (
                aggs["top_interactions"]
                .query("feat_i == @feat and feat_j != @feat")
                .sort_values("mean_abs_interaction", ascending=False)
                .head(1)
            )
            partner_feat = partner["feat_j"].iloc[0] if len(partner) else feat
            fig, ax = plt.subplots(figsize=(7, 5))
            plot_shap_dependence(shap_long, X_df, feat, partner_feat, ax, title=f"{label}: {feat} dependence")
            _save(fig, out_dir / f"shap_dependence_{feat}.png")

    headline_top = aggs["global_importance"].head(15).to_dict(orient="records")
    headline_inter = (
        aggs["top_interactions"]
        .query("feat_i != feat_j")
        .sort_values("mean_abs_interaction", ascending=False)
        .head(10)
        .to_dict(orient="records")
        if len(aggs["top_interactions"])
        else []
    )
    mean_depth = float("nan")
    if len(tree_structures):
        col = "Depth" if "Depth" in tree_structures.columns else "node_depth"
        mean_depth = float(tree_structures[col].mean())
    top_pair = f"{headline_inter[0]['feat_i']} \u00d7 {headline_inter[0]['feat_j']}" if headline_inter else "—"
    top_feat = headline_top[0]["feature"] if headline_top else "—"
    top_share = (
        float(headline_top[0]["mean_abs_shap"]) / float(aggs["global_importance"]["mean_abs_shap"].sum())
        if headline_top and aggs["global_importance"]["mean_abs_shap"].sum() > 0
        else float("nan")
    )
    stats = {
        "methodology": methodology,
        "mean_tree_depth": mean_depth,
        "top_shap_features": [r["feature"] for r in headline_top],
        "top_shap_table": headline_top,
        "top_shap_feature": top_feat,
        "top_shap_share": top_share,
        "rank_stability_rho": aggs["rank_stability_rho"],
        "top_interactions": headline_inter,
        "top_interaction_pair": top_pair,
    }
    (out_dir / "tree_story_stats.json").write_text(json.dumps(stats, indent=2, default=str), encoding="utf-8")

In [ ]:
def process_entry(repo: Path, entry: dict, out_root: Path, skip_existing: bool) -> bool:
    eid = entry_id(entry)
    out_dir = out_root / eid
    if skip_existing and (out_dir / "tree_story_stats.json").exists():
        print(f"[{eid}] skip (already built)")
        return True

    method_short = entry["method"]
    method_long = METHOD_MAP.get(method_short, method_short)
    if method_long not in TREE_DATA_PREP:
        print(f"[{eid}] unknown method '{method_short}' / '{method_long}'— skipping")
        return False

    summary_csv = entry.get("summary_csv", "")
    params_path = (repo / entry["results_dir"] / summary_csv).resolve() if summary_csv.endswith(".json") else None
    if params_path is None or not params_path.exists():
        print(f"[{eid}] no tuned params at {params_path}— skipping")
        return False
    params = json.loads(params_path.read_text(encoding="utf-8"))

    print(f"[{eid}] loading X,y from {DATA_PATH}...")
    X, y, dates, feature_names = load_xy(method_long, DATA_PATH)
    train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
    refit_frequency = TREE_DATA_PREP[method_long]["refit_frequency"]
    end = TREE_DIAG_END if TREE_DIAG_END > 0 else None

    n_oos_target = (end if end else len(X)) - train_win
    print(
        f"[{eid}] X={X.shape}  feats={len(feature_names)}  "
        f"train_win={train_win}  refit={refit_frequency}  n_oos={n_oos_target}"
    )

    shap_long, interactions_long, tree_structures, yhats = walk_forward_shap(
        method_long,
        params,
        X,
        y,
        dates,
        feature_names,
        train_win=train_win,
        refit_frequency=refit_frequency,
        interaction_stride=INTERACTION_STRIDE,
        end=end,
    )

    aggs = aggregate_shap(shap_long, interactions_long, feature_names)

    n_end = end if end else len(X)
    X_oos_df = pd.DataFrame(X[train_win:n_end], columns=feature_names)

    methodology = {
        "method": method_long,
        "params_source": str(params_path.relative_to(repo)),
        "refit_frequency": refit_frequency,
        "interaction_stride": INTERACTION_STRIDE,
        "train_window_days": TRAIN_WINDOW_DAYS,
        "n_oos_rows": int(np.isfinite(yhats).sum()),
        "n_features": len(feature_names),
        "n_interaction_rows": int(interactions_long["date"].nunique()) if len(interactions_long) else 0,
        "shap_target_scale": "adjusted",
        "caveats": [
            "SHAP attributes prediction-scale, not loss-scale.",
            "Pooled across heterogeneous estimators in the walk-forward sequence.",
            "Correlated features make individual attributions ambiguous (Aas et al. 2021); see group-bar plot.",
            "18:30/Sunday-open intraday pattern is an upstream aggregation artifact "
            "and is not used as a narrative anchor.",
        ],
    }

    label = f"{entry['method']} / {entry['feature_set']} / {entry['config']}"
    render_tree_diagnostics(
        out_dir,
        label,
        tree_structures,
        shap_long,
        interactions_long,
        aggs,
        X_oos_df,
        feature_names,
        methodology,
    )
    print(f"[{eid}] -> {out_dir}")
    return True

In [ ]:
manifest_path = Path(MANIFEST)
if not manifest_path.is_absolute():
    manifest_path = REPO / manifest_path
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

out_root = Path(OUT_ROOT)
if not out_root.is_absolute():
    out_root = REPO / out_root

if ENTRY:
    targets = [ENTRY]
else:
    targets = TREE_ENTRIES

entries = [e for e in manifest["entries"] if entry_id(e) in targets]
if not entries:
    raise ValueError(f"no entries matched {targets}")

n_ok = sum(process_entry(REPO, e, out_root, SKIP_EXISTING) for e in entries)
print(f"\nDone: {n_ok}/{len(entries)} tree-diagnostic bundles built -> {out_root}")